# QLoRA 학습 — 충남대 Q&A 시스템

Colab T4 GPU에서 실행합니다.

- Base: `Qwen/Qwen2.5-7B-Instruct` (4bit NF4)
- QLoRA: r=16, alpha=32, dropout=0.05, target=q/k/v/o_proj
- 학습 데이터: HF Hub에서 다운로드
- 학습 완료 후 어댑터를 HF Hub에 업로드

> **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 실행하세요.

## 1. 환경 설정

In [ ]:
%%time
!pip install -q transformers accelerate bitsandbytes peft datasets huggingface_hub

In [ ]:
import torch
import random

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. HF Hub 로그인 & 데이터 다운로드

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUTPUT = "/content/drive/MyDrive/cnu_lora_adapter"
import os
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Drive 저장 경로: {DRIVE_OUTPUT}")

In [ ]:
from huggingface_hub import login, snapshot_download

# 아래 실행하면 토큰 입력창이 나옵니다
login()

In [ ]:
HF_REPO = "adoveflash/cnu-qa-system"

import os

# train.jsonl 다운로드
TRAIN_PATH = "data/qa/train.jsonl"
if not os.path.exists(TRAIN_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/qa/train.jsonl"],
    )
print(f"학습 데이터: {TRAIN_PATH}")

import json
with open(TRAIN_PATH) as f:
    train_data = [json.loads(line) for line in f if line.strip()]
print(f"학습 샘플 수: {len(train_data)}")

## 3. 모델 & 토크나이저 로드

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4bit NF4 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[2/2] 모델 로드 (4bit NF4)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

## 4. LoRA 설정 & 적용

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. 데이터셋 준비

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "당신은 충남대학교 학내 정보 안내 도우미입니다. "
    "주어진 참고 자료를 바탕으로 정확하게 답변하세요. "
    "참고 자료에 없는 내용은 '확인되지 않은 정보입니다'라고 답하세요."
)
MAX_LENGTH = 256

random.seed(SEED)
random.shuffle(train_data)

# chat template 적용
texts = []
for qa in train_data:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": qa["question"]},
        {"role": "assistant", "content": qa["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    texts.append(text)

print(f"샘플 텍스트 (첫 번째):\n{texts[0][:300]}...")


def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


dataset = Dataset.from_dict({"text": texts})
dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f"데이터셋 크기: {len(dataset)}")

## 6. 학습

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback
from huggingface_hub import HfApi

CKPT_DIR = f"{DRIVE_OUTPUT}/checkpoints"
LOCAL_OUTPUT = "models/lora_adapter"
HF_REPO = "adoveflash/cnu-qa-system"


class HubUploadCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        print(f"
에폭 {state.epoch:.0f} → HF Hub 백업 중...")
        try:
            api = HfApi()
            ckpts = sorted(
                [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1])
            )
            if ckpts:
                latest_path = os.path.join(CKPT_DIR, ckpts[-1])
                api.upload_folder(
                    folder_path=latest_path,
                    path_in_repo="models/lora_adapter",
                    repo_id=HF_REPO,
                )
                print(f"백업 완료: {ckpts[-1]}")
        except Exception as e:
            print(f"백업 실패 (학습은 계속): {e}")


training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    seed=SEED,
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    callbacks=[HubUploadCallback()],
)

# 체크포인트에서 이어서 학습
resume_ckpt = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_ckpt = os.path.join(CKPT_DIR, sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"체크포인트에서 재개: {resume_ckpt}")

print("학습 시작")
trainer.train(resume_from_checkpoint=resume_ckpt)
print("학습 완료!")

# 최종 어댑터 저장
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
model.save_pretrained(LOCAL_OUTPUT)
tokenizer.save_pretrained(LOCAL_OUTPUT)
print(f"어댑터 저장: {LOCAL_OUTPUT}")

## 7. 어댑터 저장 & HF Hub 업로드

In [ ]:
# 로컬 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"어댑터 저장 완료: {OUTPUT_DIR}")

# 저장된 파일 확인
import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size / 1024**2:.1f} MB")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    path_in_repo="models/lora_adapter",
    repo_id=HF_REPO,
)
print(f"HF Hub 업로드 완료: {HF_REPO}/models/lora_adapter")

## 8. 추론 테스트

In [ ]:
# LoRA 모델로 추론 테스트
model.eval()

test_questions = [
    "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
    "수강신청은 언제 하나요?",
    "장학금 신청은 어떻게 하나요?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

## 완료

어댑터가 HF Hub에 업로드되었습니다.
`submission.ipynb`에서 자동으로 다운로드하여 사용합니다.